# Preprocessing Pipeline — ALPR Dataset

This notebook covers Parts 3–7 of the pipeline:
- **Part 3 — Quality Check**: corruption detection, blur filtering, bbox validation
- **Part 4 — Harmonization**: convert all annotations to unified YOLO format
- **Part 5 — Preprocessing**: apply configurable transform pipeline (denoise → CLAHE → gamma → bilateral → sharpen → letterbox)
- **Part 6 — Statistics**: compute pre/post stats and before/after comparisons
- **Part 7 — Splitting**: stratified train/val/test split

All stages are driven by `configs/preprocessing_config.yaml`.

**Output:** Preprocessed images + YOLO labels written to `data/processed/`.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

# Walk up to find project root (handles kernels with different CWD)
_candidate = Path.cwd().resolve()
for _parent in [_candidate] + list(_candidate.parents):
    if (_parent / "src" / "alpr_dataset").is_dir():
        PROJECT_ROOT = _parent
        break
else:
    PROJECT_ROOT = _candidate
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")

---
## 1. Load Configuration

In [ ]:
from alpr_dataset.config import PipelineConfig
from alpr_dataset.logging_setup import setup_logging

config = PipelineConfig.load(
    PROJECT_ROOT / "configs" / "pipeline_config.yaml",
    PROJECT_ROOT / "configs" / "datasets.yaml",
)
prep_config = config.preprocessing_config(
    PROJECT_ROOT / "configs" / "preprocessing_config.yaml"
)
logger = setup_logging(config.logs_dir, name="alpr_dataset")

print(f"Datasets: {[s.name for s in config.datasets]}")
print(f"Target size: {prep_config.target_size}")
print(f"Steps enabled: {[s.name for s in prep_config.steps if s.enabled]}")
print(f"Split ratio: T={prep_config.split.train_ratio} V={prep_config.split.val_ratio} Te={prep_config.split.test_ratio}")

---
## 2. Load Annotations (all datasets)

In [ ]:
from alpr_dataset.annotations.loader import load_dataset_annotations

all_annotations = {}
for spec in config.datasets:
    all_annotations[spec.name] = load_dataset_annotations(spec)
    print(f"{spec.name}: {len(all_annotations[spec.name])} annotations loaded")

print(f"\nTotal annotated images: {sum(len(v) for v in all_annotations.values())}")

---
## 3. Part 3 — Quality Check

For each dataset: detect corrupt images, compute blur scores, validate bbox coordinates.

In [ ]:
from alpr_dataset.eda.quality import compute_quality_report

quality_reports = {}
for spec in config.datasets:
    qr = compute_quality_report(
        spec, blur_threshold=config.blur_threshold
    )
    quality_reports[spec.name] = qr
    print(f"\n{'='*50}\n{spec.name}\n{'='*50}")
    print(f"  Corrupt images: {len(qr.corrupt_images)}")
    print(f"  Blurry images (VoF < {config.blur_threshold}): {len(qr.blurry_images)}")
    print(f"  Invalid bboxes: {qr.invalid_bbox_count}")
    print(f"  Filtered out: {qr.total_filtered}")

### 3.1 Visualise Blurry / Corrupt Samples

In [ ]:
import random
from alpr_dataset.io_utils import safe_read_image
from alpr_dataset.utils.viz_utils import bgr_to_rgb

rng = random.Random(0)
for spec in config.datasets:
    qr = quality_reports[spec.name]
    blurry = list(qr.blurry_images)
    if not blurry:
        print(f"{spec.name}: No blurry images to show.")
        continue
    
    sample = rng.sample(blurry, min(4, len(blurry)))
    fig, axes = plt.subplots(2, 2, figsize=(8, 8))
    for idx, img_path in enumerate(sample):
        img = safe_read_image(img_path)
        axes.flat[idx].imshow(bgr_to_rgb(img) if img is not None else None)
        axes.flat[idx].set_title(f"Blurry: {img_path.name}", fontsize=8)
        axes.flat[idx].axis("off")
    fig.suptitle(f"{spec.name}: Blurry Samples (VoF < {config.blur_threshold})")
    fig.tight_layout(); plt.show()

---
## 4. Part 4 — Harmonization

Converts all annotations to unified YOLO format in `data/processed/unified/`.

In [ ]:
from alpr_dataset.harmonization.harmonizer import harmonize_dataset

unified_dir = config.processed_dir / "unified"
unified_dir.mkdir(parents=True, exist_ok=True)

for spec in config.datasets:
    result = harmonize_dataset(
        spec,
        all_annotations[spec.name],
        output_dir=unified_dir,
        class_map=spec.class_map or {},
    )
    print(f"{spec.name}: {result.n_written} YOLO labels written to {result.output_dir}")

In [ ]:
# Verify harmonized output
expected_classes = set()
for spec in config.datasets:
    if spec.class_map:
        expected_classes.update(spec.class_map.values())
    (
        unified_dir / spec.name / "images"
    ).mkdir(parents=True, exist_ok=True)
    
n_labels = len(list(unified_dir.glob("*/*.txt")))
n_class_yamls = len(list(unified_dir.glob("dataset_*.yaml")))
print(f"Unified YOLO labels: {n_labels}")
print(f"Dataset YAML files: {n_class_yamls}")
print(f"Expected classes: {sorted(expected_classes)}")

---
## 5. Part 5 — Preprocessing Pipeline

Apply the configured transform pipeline. Steps (from config):

| Step | Enabled | Params |
|------|---------|--------|
{% for step in prep_config.steps %}| {{ step.name }} | {{ step.enabled }} | {{ step.params }} |
{% endfor %}

> **Note:** The Jinja loop above doesn't render in Jupyter. See the cell output below for the actual config.

In [ ]:
from alpr_dataset.preprocessing.pipeline import PreprocessingPipeline

pipeline = PreprocessingPipeline(prep_config)

print("Pipeline steps (in order):")
for i, step in enumerate(prep_config.steps):
    if not step.enabled:
        continue
    print(f"  {i+1}. {step.name}  enabled={step.enabled}  params={step.params}")
print(f"\nTarget size: {prep_config.target_size}")


### 5.1 Interactive Transform Demo

Pick a sample image and visualise what each transform does.

In [ ]:
import cv2
import numpy as np
from alpr_dataset.io_utils import list_images

# Find first available image
sample_img_path = None
for spec in config.datasets:
    images = list_images(spec.root)
    if images:
        sample_img_path = images[0]
        break

if sample_img_path is None:
    raise RuntimeError("No images found in any dataset!")

original = safe_read_image(sample_img_path)
print(f"Sample: {sample_img_path} \u2014 shape={original.shape}" if original is not None else "Failed to load")

# Apply each enabled step independently using STEP_REGISTRY
from alpr_dataset.preprocessing.pipeline import STEP_REGISTRY

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes[0,0].imshow(bgr_to_rgb(original))
axes[0,0].set_title("Original"); axes[0,0].axis("off")

ax_idx = 1
seen_names = ["Original"]
for step in prep_config.steps:
    if not step.enabled or step.name in ("letterbox",):
        continue
    fn = STEP_REGISTRY.get(step.name)
    if fn is None:
        continue
    params = dict(step.params)
    if step.name in ("resize", "letterbox") and "target_size" not in params:
        params["target_size"] = prep_config.target_size
    transformed = fn(original.copy(), **params)
    r, c = divmod(ax_idx, 3)
    axes[r,c].imshow(bgr_to_rgb(transformed))
    axes[r,c].set_title(step.name, fontsize=9); axes[r,c].axis("off")
    ax_idx += 1
    seen_names.append(step.name)

for a in axes.flat[ax_idx:]:
    a.axis("off")
fig.suptitle(f"Per-step transforms: {sample_img_path.name}", fontsize=12)
fig.tight_layout(); plt.show()


### 5.2 Before / After Comparison

Original vs full pipeline output on random samples.

In [ ]:
rng = random.Random(42)
all_images = []
for spec in config.datasets:
    all_images.extend(list_images(spec.root))

sample_paths = rng.sample(all_images, min(6, len(all_images)))

fig, axes = plt.subplots(3, 2, figsize=(10, 12))
for idx, img_path in enumerate(sample_paths):
    img = safe_read_image(img_path)
    if img is None:
        continue
    processed = pipeline.apply(img)
    
    ax = axes.flat[idx]
    comparison = np.hstack([img, processed])
    ax.imshow(bgr_to_rgb(comparison))
    ax.set_title(f"Before (L) / After (R) \u2014 {img_path.name}", fontsize=8)
    ax.axis("off")
fig.suptitle("Preprocessing: Before vs After", fontsize=14)
fig.tight_layout(); plt.show()


### 5.3 Run on full dataset

In [ ]:
from tqdm import tqdm
from alpr_dataset.annotations.loader import load_dataset_annotations
from alpr_dataset.annotations.writer import convert_to_yolo

processed_root = config.processed_dir / "preprocessed"

for spec in config.datasets:
    images = list_images(spec.root)
    print(f"Processing {spec.name} ({len(images)} images)...")
    
    out_img_dir = processed_root / spec.name / "images"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    
    # Load annotations and convert to YOLO
    annots = load_dataset_annotations(spec)
    out_lbl_dir = processed_root / spec.name / "labels"
    out_lbl_dir.mkdir(parents=True, exist_ok=True)
    
    n_ok = 0
    for img_path in tqdm(images, desc=spec.name):
        img = safe_read_image(img_path)
        if img is None:
            continue
        processed = pipeline.apply(img)
        cv2.imencode(".jpg", processed)[1].tofile(str(out_img_dir / img_path.name))
        
        # Write YOLO label from harmonized annotations
        if img_path.stem in annots:
            yolo_lines = convert_to_yolo(annots[img_path.stem], *processed.shape[:2])
            with open(out_lbl_dir / f"{img_path.stem}.txt", "w") as f:
                f.writelines(yolo_lines)
        n_ok += 1
    
    print(f"  Written: {n_ok} images + labels to {out_img_dir}")


---
## 6. Part 6 — Statistics

Pre/post comparison: resolution, brightness, contrast, blur, entropy.

In [ ]:
from alpr_dataset.inspection.image_stats import compute_image_stats, batch_compute_stats
from alpr_dataset.preprocessing.stats_generator import compute_pre_post_stats

stats_dir = config.reports_dir / "preprocessing_stats"
stats_dir.mkdir(parents=True, exist_ok=True)

for spec in config.datasets:
    print(f"\n{spec.name}:")
    pre_images = list_images(spec.root)
    post_images = list_images(processed_root / spec.name / "images")
    
    pre_stats = batch_compute_stats(pre_images)
    post_stats = batch_compute_stats(post_images)
    
    pre_valid = [s for s in pre_stats if not s.is_corrupted]
    post_valid = [s for s in post_stats if not s.is_corrupted]
    
    print(f"  Pre:  {len(pre_valid)} valid images")
    print(f"  Post: {len(post_valid)} valid images")
    
    for attr, label in [
        ("brightness_mean", "Brightness"),
        ("contrast_std", "Contrast"),
        ("blur_score", "Blur"),
        ("entropy", "Entropy"),
    ]:
        pre_vals = [getattr(s, attr) for s in pre_valid]
        post_vals = [getattr(s, attr) for s in post_valid]
        if pre_vals:
            print(f"  {label}: Pre={sum(pre_vals)/len(pre_vals):.2f}  Post={sum(post_vals)/len(post_vals):.2f}")

In [ ]:
# Before/after comparative plots
spec = config.datasets[0]
pre_images = list_images(spec.root)
post_images = list_images(processed_root / spec.name / "images")
pre_stats = batch_compute_stats(pre_images)
post_stats = batch_compute_stats(post_images)
pre_valid = [s for s in pre_stats if not s.is_corrupted]
post_valid = [s for s in post_stats if not s.is_corrupted]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
metrics = [
    ("brightness_mean", "Brightness", "#f4d35e"),
    ("contrast_std", "Contrast", "#ee6c4d"),
    ("blur_score", "Blur", "#3d5a80"),
    ("sharpness_score", "Sharpness", "#98c1d9"),
    ("entropy", "Entropy", "#293241"),
]

for idx, (attr, label, color) in enumerate(metrics):
    r, c = divmod(idx, 3)
    pre_vals = [getattr(s, attr) for s in pre_valid]
    post_vals = [getattr(s, attr) for s in post_valid]
    
    axes[r,c].hist(pre_vals, bins=40, alpha=0.6, color=color, label="Pre")
    axes[r,c].hist(post_vals, bins=40, alpha=0.6, color="#e63946", label="Post")
    axes[r,c].set_title(label)
    axes[r,c].legend(fontsize=8)

axes[1,2].axis("off")
fig.suptitle(f"{spec.name}: Pre / Post Comparison", fontsize=14)
fig.tight_layout(); plt.show()

---
## 7. Part 7 — Stratified Split

Split preprocessed data into train/val/test sets (default 70/15/15) with stratification by class.

In [ ]:
from alpr_dataset.splitting.splitter import stratified_split, write_split_manifests
from alpr_dataset.config import SplitConfig

split_cfg = config.split_config(
    PROJECT_ROOT / "configs" / "preprocessing_config.yaml"
)

for spec in config.datasets:
    print(f"\n{spec.name}:")
    image_dir = processed_root / spec.name / "images"
    label_dir = processed_root / spec.name / "labels"

    if not label_dir.is_dir():
        label_dir = unified_dir / spec.name

    all_annotations = load_dataset_annotations(spec)
    annotations = list(all_annotations.values())

    result = stratified_split(annotations, split_cfg)
    summary = result.summary()
    print(f"  Train: {summary[\"n_train\"]}, Val: {summary[\"n_val\"]}, Test: {summary[\"n_test\"]}")

    out_dir = config.processed_dir / "split" / spec.name
    write_split_manifests(result, out_dir)
    print(f"  Manifests -> {out_dir}")


In [ ]:
# Visualise split distribution
spec = config.datasets[0]

all_annotations = load_dataset_annotations(spec)
annotations = list(all_annotations.values())

split_cfg = config.split_config(
    PROJECT_ROOT / "configs" / "preprocessing_config.yaml"
)
result = stratified_split(annotations, split_cfg)
summary = result.summary()

counts = {
    "train": summary["n_train"],
    "val": summary["n_val"],
    "test": summary["n_test"],
}

if any(counts.values()):
    fig, ax = plt.subplots(figsize=(5, 4))
    colors = ["#3d5a80", "#ee6c4d", "#98c1d9"]
    bars = ax.bar(counts.keys(), counts.values(), color=colors, edgecolor="white")
    ax.set_title(f"{spec.name}: Train / Val / Test Split")
    ax.set_ylabel("Number of images")
    for bar, val in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                str(val), ha="center", fontsize=10)
    fig.tight_layout(); plt.show()

    total = sum(counts.values())
    print(f"Total: {total}  |  Ratios: T={counts[\"train\"]/total:.1%} V={counts[\"val\"]/total:.1%} Te={counts[\"test\"]/total:.1%}")


---
## 8. Run Full Pipeline (single command)

The equivalent of `scripts/run_full_pipeline.py` encompassing all stages.

In [ ]:
print("All stages complete.")
print(f"\nOutput directories:")
print(f"  Unified YOLO:   {unified_dir}")
print(f"  Preprocessed:   {processed_root}")
print(f"  Split:          {config.processed_dir / \"split\"}")
print(f"  Reports:        {config.reports_dir}")
print(f"  Logs:           {config.logs_dir}")


In [ ]:
# Validate output structure
print("Split directory structure:")
for p in sorted((config.processed_dir / "split").glob("*/*/*")):
    print(f"  {p.relative_to(config.processed_dir / 'split')}")

print(f"\nPreprocessed images total: {len(list(processed_root.glob('*/*/*.*')))} files")